# Task 2 · Part 1 — From scanned documents and recorded interviews to searchable text

**Guiding question:** Can scanned documents and audio interviews be converted into a structured, searchable text archive?

This notebook builds the two corpora of the project and extracts their text with open models running locally: PaddleOCR for documents and faster-whisper for audio. It then pseudonymizes, cleans, explores, validates and exports the results that feed the evaluation (Part 2) and the dashboard (Part 3).

| Parameter | Value |
|---|---|
| **Extraction date** | 24–25 October 2026 |
| **Documents** | 25 Peruvian documents: 8 printed pages from *El Peruano*, Normas Legales, 24/10/2026 (legal texts, not subject to copyright under Legislative Decree 822, art. 9); 8 photographs and 9 handwritten notes taken by the author in Lima |
| **Audio** | 6 research interviews conducted by the author with signed informed consent for academic use; 36 minutes cut into 72 clips of 30 s |
| **OCR** | PaddleOCR 3.7.0 — PP-OCRv5 mobile detection + Latin mobile recognition, CPU |
| **ASR** | faster-whisper `small` (int8, CPU) with Silero VAD; diarization with WeSpeaker embeddings (`voz.py`, Session 13) |
| **Environment** | Course environment: `uv sync --group docai --group audio` (Python 3.12) |

**Ethics.** Original images and audio are kept in `datos/raw/`, which is excluded from Git. Only reviewed and pseudonymized text is published. The interviews' audio is never redistributed.

In [1]:
from pathlib import Path

import paddleocr
import faster_whisper

# Raw corpus folders (excluded from Git through .gitignore).
RAW = Path("datos/raw")

for carpeta in ["documentos", "audio"]:
    archivos = sorted(p.name for p in (RAW / carpeta).iterdir())
    print(f"{carpeta}: {len(archivos)} files")
    for nombre in archivos:
        print("  -", nombre)

print("PaddleOCR version:", paddleocr.__version__)

documentos: 9 files
  - doc_009.jpeg
  - doc_010.jpeg
  - doc_011.jpeg
  - doc_012.jpeg
  - doc_013.jpeg
  - doc_017.jpeg
  - doc_018.jpeg
  - doc_019.jpeg
  - elperuano_20261024.pdf.pdf
audio: 6 files
  - entrevista_01.mp4
  - entrevista_02.mp4
  - entrevista_03.m4a
  - entrevista_04.mp4
  - entrevista_05.mp4
  - entrevista_06.mp4
PaddleOCR version: 3.7.0


## 1. Printed corpus: rendering pages from *El Peruano*

The official gazette publishes each day's legal norms as a single PDF. Eight pages, evenly spaced across the issue and skipping the cover and table of contents, are rendered at 200 dpi. Each page becomes one printed document (`doc_001`–`doc_008`), and its page number is recorded so that the selection can be reproduced.

In [2]:
import fitz  # PyMuPDF, installed with the docai group
import numpy as np

carpeta_docs = RAW / "documentos"

# Fix the duplicated extension produced by Windows when the file was renamed.
pdf_doble = carpeta_docs / "elperuano_20261024.pdf.pdf"
pdf_path = carpeta_docs / "elperuano_20261024.pdf"
if pdf_doble.exists():
    pdf_doble.rename(pdf_path)

pdf = fitz.open(pdf_path)
print("Total pages in the issue:", pdf.page_count)

# Select eight evenly spaced pages, skipping the cover and the table of contents.
paginas = [int(x) for x in np.linspace(4, pdf.page_count - 1, 8)]

for i, num in enumerate(paginas, start=1):
    pagina = pdf[num - 1]
    imagen = pagina.get_pixmap(dpi=200)  # 200 dpi keeps the text sharp for OCR
    destino = carpeta_docs / f"doc_{i:03d}.png"
    imagen.save(destino)
    # Show the first characters of the page to identify the issuing institution.
    inicio = " ".join(pagina.get_text().split())[:90]
    print(f"doc_{i:03d} <- page {num}: {inicio}")

Total pages in the issue: 116
doc_001 <- page 4: 4 NORMAS LEGALES Sábado 24 de octubre de 2026 El Peruano / Res. N° 3335-2026-MP-FN.- Dan p
doc_002 <- page 19: 19 NORMAS LEGALES Sábado 24 de octubre de 2026 El Peruano / - SUNARP, inscriba a favor de 
doc_003 <- page 35: 35 NORMAS LEGALES Sábado 24 de octubre de 2026 El Peruano / indica que la Expropiación con
doc_004 <- page 51: 51 NORMAS LEGALES Sábado 24 de octubre de 2026 El Peruano / SE RESUELVE: Artículo 1.- Se a
doc_005 <- page 67: 67 NORMAS LEGALES Sábado 24 de octubre de 2026 El Peruano / las características y especiﬁ 
doc_006 <- page 83: 83 NORMAS LEGALES Sábado 24 de octubre de 2026 El Peruano / PROCURADURÍA GENERAL DEL ESTAD
doc_007 <- page 99: 99 NORMAS LEGALES Sábado 24 de octubre de 2026 El Peruano / VISTOS: El oﬁ cio Nº 005982-20
doc_008 <- page 115: 115 NORMAS LEGALES Sábado 24 de octubre de 2026 El Peruano / las vías propuestas, en confo


## 2. Optical character recognition with PaddleOCR

The OCR settings follow the Session 10 lecture: page rotation, unwarping and line rotation are disabled. Two technical decisions were required on this machine. First, the oneDNN accelerator was disabled because it raises a known `NotImplementedError` with PaddlePaddle 3.x on Windows. Second, the default server models needed four to five minutes per printed page on CPU, so the lightweight mobile models were adopted for the whole corpus. A raw backup with one row per detected line and its confidence score is saved in `datos/raw/documentos/ocr_crudo/`.

In [7]:
import time

import pandas as pd
from paddleocr import PaddleOCR

# Same settings as the Session 10 lecture: no page rotation, unwarping or line rotation.
OPCIONES = dict(
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
)
# Lightweight "mobile" models: the default server models took ~4-5 min per page on this CPU.
# oneDNN acceleration fails on Windows CPU with PaddlePaddle 3.x, so it is disabled.
ocr = PaddleOCR(
    text_detection_model_name="PP-OCRv5_mobile_det",
    text_recognition_model_name="latin_PP-OCRv5_mobile_rec",  # Latin-script model, covers Spanish
    device="cpu",
    enable_mkldnn=False,
    **OPCIONES,
)
MODELO_OCR = f"PaddleOCR {paddleocr.__version__} (PP-OCRv5 mobile det + latin mobile rec, CPU)"

carpeta_crudo = carpeta_docs / "ocr_crudo"
carpeta_crudo.mkdir(exist_ok=True)


def tipo_documento(numero: int) -> str:
    """Map the reserved id ranges to the three document types."""
    if numero <= 8:
        return "impreso"
    if numero <= 16:
        return "foto"
    return "manuscrito"


def fuente_documento(numero: int) -> str:
    """Return the origin and licence of each document."""
    if numero <= 8:
        return (f"Diario Oficial El Peruano, Normas Legales 24/10/2026, p. {paginas[numero - 1]} "
                "(legal text, not subject to copyright: Legislative Decree 822, art. 9)")
    return "Own photograph by the author, Lima, October 2026 (no personal data of private individuals)"


registros = []
archivos = sorted(carpeta_docs.glob("doc_*.*"))
for ruta in archivos:
    numero = int(ruta.stem.split("_")[1])
    inicio = time.perf_counter()
    resultado = ocr.predict(str(ruta))[0]
    segundos = time.perf_counter() - inicio

    # Raw backup: one row per detected line with its confidence score.
    lineas = pd.DataFrame({"texto": resultado["rec_texts"], "confianza": resultado["rec_scores"]})
    lineas.to_csv(carpeta_crudo / f"{ruta.stem}.csv", index=False)

    registros.append({
        "id": ruta.stem,
        "archivo": ruta.as_posix(),
        "tipo": tipo_documento(numero),
        "fuente": fuente_documento(numero),
        "texto_ocr": " ".join(resultado["rec_texts"]),
        "modelo_ocr": MODELO_OCR,
        "texto_referencia": "",
        "confianza_media": round(lineas["confianza"].mean(), 3),
        "segundos": round(segundos, 1),
    })
    print(f"{ruta.stem}: {len(lineas)} lines, {segundos:.1f} s")

documentos = pd.DataFrame(registros)
documentos[["id", "tipo", "confianza_media", "segundos"]]

Creating model: ('PP-OCRv5_mobile_det', None, None)
Using official model (PP-OCRv5_mobile_det), the model files will be automatically downloaded and saved in `C:\Users\Ana Karen\.paddlex\official_models\PP-OCRv5_mobile_det`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('latin_PP-OCRv5_mobile_rec', None, None)
Using official model (latin_PP-OCRv5_mobile_rec), the model files will be automatically downloaded and saved in `C:\Users\Ana Karen\.paddlex\official_models\latin_PP-OCRv5_mobile_rec`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

doc_001: 126 lines, 73.9 s
doc_002: 267 lines, 119.8 s
doc_003: 184 lines, 198.9 s
doc_004: 146 lines, 133.5 s
doc_005: 184 lines, 177.3 s
doc_006: 140 lines, 139.0 s
doc_007: 143 lines, 165.5 s
doc_008: 161 lines, 90.7 s
doc_009: 9 lines, 6.4 s
doc_010: 38 lines, 12.9 s
doc_011: 125 lines, 42.3 s
doc_012: 13 lines, 7.3 s
doc_013: 19 lines, 14.6 s
doc_014: 50 lines, 27.0 s
doc_015: 25 lines, 21.1 s
doc_016: 56 lines, 31.0 s
doc_017: 19 lines, 7.7 s
doc_018: 44 lines, 18.6 s
doc_019: 22 lines, 11.7 s
doc_020: 47 lines, 24.7 s
doc_021: 37 lines, 18.4 s
doc_022: 45 lines, 29.4 s
doc_023: 19 lines, 11.4 s
doc_024: 35 lines, 17.8 s
doc_025: 77 lines, 41.1 s


,id,tipo,confianza_media,segundos
0,doc_001,impreso,0.924,73.9
1,doc_002,impreso,0.990,119.8
2,doc_003,impreso,0.970,198.9
3,doc_004,impreso,0.976,133.5
4,doc_005,impreso,0.955,177.3
5,doc_006,impreso,0.979,139.0
6,doc_007,impreso,0.982,165.5
7,doc_008,impreso,0.915,90.7
8,doc_009,foto,0.905,6.4
9,doc_010,foto,0.817,12.9


## 3. Speech recognition: clipping, VAD, transcription and diarization

Each interview is cut into 30-second clips starting at minute one, so that introductions and the reading of the consent form are excluded. Short clips also make a manual reference transcription feasible in Part 2. Every clip is transcribed with faster-whisper `small`, the baseline required by the assignment, with voice activity detection enabled to prevent hallucinations during silences. Speakers are then identified with the `diarizar` function from Session 13, which clusters voice embeddings. Subtitles (SRT) and readable transcripts (TXT) are saved in `datos/raw/audio/salidas/`.

In [9]:
import sys

import soundfile as sf
from faster_whisper import WhisperModel

# Reuse the Session 13 helpers (audio loading, diarization, SRT/TXT export).
sys.path.append(r"C:\Users\Ana Karen\Documents\GitHub\Data-Science-Python\sessions\13-audio-a-texto")
import voz

carpeta_audio = RAW / "audio"
carpeta_clips = carpeta_audio / "clips"
carpeta_salidas = carpeta_audio / "salidas"
carpeta_clips.mkdir(exist_ok=True)
carpeta_salidas.mkdir(exist_ok=True)

INICIO_SEG = 60            # skip the first minute (introductions and consent reading)
DURACION_CLIP = 30         # 30-second clips keep manual reference transcription feasible
CLIPS_POR_ENTREVISTA = 12  # 6 interviews x 12 clips = 72 clips (~36 min)

# 1) Cut each interview into fixed-length clips (16 kHz mono WAV).
clips = []
for ruta in sorted(carpeta_audio.glob("entrevista_*.*")):
    audio = voz.cargar_audio(ruta)
    total = len(audio) / voz.SR
    for k in range(CLIPS_POR_ENTREVISTA):
        a = INICIO_SEG + k * DURACION_CLIP
        b = min(a + DURACION_CLIP, total)
        if b - a < 10:
            break
        clip_id = f"clip_{len(clips) + 1:03d}"
        destino = carpeta_clips / f"{clip_id}.wav"
        sf.write(destino, audio[int(a * voz.SR):int(b * voz.SR)], voz.SR)
        clips.append({"id": clip_id, "entrevista": ruta.stem, "inicio_seg": a,
                      "archivo_audio": destino.as_posix(), "duracion_seg": round(b - a, 1)})
    print(f"{ruta.stem}: {total / 60:.1f} min")

# 2) Transcribe with faster-whisper small (baseline required by the assignment) and VAD on.
modelo_asr = WhisperModel("small", device="cpu", compute_type="int8")
MODELO_ASR = "faster-whisper small, int8, CPU, VAD on"

entrevista_01: 13.3 min
entrevista_02: 13.4 min
entrevista_03: 10.3 min
entrevista_04: 41.3 min
entrevista_05: 44.8 min
entrevista_06: 35.7 min


In [10]:
def unir_turnos(segmentos):
    """Merge consecutive segments of the same speaker and add [HABLANTE_n] markers."""
    turnos = []
    for s in segmentos:
        if turnos and turnos[-1][0] == s.hablante:
            turnos[-1][1].append(s.texto)
        else:
            turnos.append([s.hablante, [s.texto]])
    return " ".join(f"[HABLANTE_{h[1:]}] {' '.join(t)}" for h, t in turnos)


filas_asr = []
for c in clips:
    audio = voz.cargar_audio(c["archivo_audio"])
    inicio = time.perf_counter()
    segs, _ = modelo_asr.transcribe(
        audio, language="es", beam_size=5, vad_filter=True,
        vad_parameters={"min_silence_duration_ms": 300},
    )
    segmentos = [voz.Segmento(round(s.start, 2), round(s.end, 2), s.text.strip()) for s in segs]
    segundos = time.perf_counter() - inicio

    # Diarization: speaker fingerprints + clustering (number of speakers estimated automatically).
    segmentos = voz.diarizar(audio, segmentos)
    (carpeta_salidas / f"{c['id']}.srt").write_text(voz.a_srt(segmentos), encoding="utf-8")
    (carpeta_salidas / f"{c['id']}.txt").write_text(voz.a_texto(segmentos), encoding="utf-8")

    filas_asr.append(c | {
        "idioma": "es (Peruvian Spanish)",
        "texto_asr": unir_turnos(segmentos),
        "modelo_asr": MODELO_ASR,
        "n_hablantes": len({s.hablante for s in segmentos}),
        "texto_referencia": "",
        "segundos_asr": round(segundos, 1),
    })
    n = filas_asr[-1]["n_hablantes"]
    print(f"{c['id']}: {c['duracion_seg']} s audio, {segundos:.0f} s processing, {n} speakers")

transcripciones = pd.DataFrame(filas_asr)
total_min = transcripciones["duracion_seg"].sum() / 60
print(f"\nTotal audio: {total_min:.1f} min in {len(transcripciones)} clips")

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 26.6MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

c:\Users\Ana Karen\Documents\GitHub\Data-Science-Python\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ana Karen\.cache\huggingface\hub\models--pyannote--wespeaker-voxceleb-resnet34-LM. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


clip_001: 30 s audio, 17 s processing, 2 speakers
clip_002: 30 s audio, 19 s processing, 3 speakers
clip_003: 30 s audio, 21 s processing, 1 speakers
clip_004: 30 s audio, 17 s processing, 2 speakers
clip_005: 30 s audio, 13 s processing, 1 speakers
clip_006: 30 s audio, 26 s processing, 2 speakers
clip_007: 30 s audio, 23 s processing, 2 speakers
clip_008: 30 s audio, 16 s processing, 2 speakers
clip_009: 30 s audio, 21 s processing, 2 speakers
clip_010: 30 s audio, 21 s processing, 2 speakers
clip_011: 30 s audio, 20 s processing, 2 speakers
clip_012: 30 s audio, 21 s processing, 2 speakers
clip_013: 30 s audio, 17 s processing, 1 speakers
clip_014: 30 s audio, 22 s processing, 1 speakers
clip_015: 30 s audio, 20 s processing, 3 speakers
clip_016: 30 s audio, 18 s processing, 2 speakers
clip_017: 30 s audio, 12 s processing, 2 speakers
clip_018: 30 s audio, 16 s processing, 1 speakers
clip_019: 30 s audio, 19 s processing, 1 speakers
clip_020: 30 s audio, 17 s processing, 2 speakers


In [4]:
sorted(p.name for p in carpeta_docs.glob("doc_*"))

['doc_001.png',
 'doc_002.png',
 'doc_003.png',
 'doc_004.png',
 'doc_005.png',
 'doc_006.png',
 'doc_007.png',
 'doc_008.png',
 'doc_009.jpeg',
 'doc_010.jpeg',
 'doc_011.jpeg',
 'doc_012.jpeg',
 'doc_013.jpeg',
 'doc_014.jpeg',
 'doc_015.jpeg',
 'doc_016.jpeg',
 'doc_017.jpeg',
 'doc_018.jpeg',
 'doc_019.jpeg',
 'doc_020.jpeg',
 'doc_021.jpeg',
 'doc_022.jpeg',
 'doc_023.jpeg',
 'doc_024.jpeg',
 'doc_025.jpeg']

In [11]:
import sys

!uv pip install plotly --python "{sys.executable}"

Using Python 3.12.14 environment at: c:\Users\Ana Karen\Documents\GitHub\Data-Science-Python\.venv
Resolved 3 packages in 501ms
 Downloaded plotly
Prepared 1 package in 4.21s
Installed 1 package in 1.66s
 + plotly==7.1.0


## 4. Pseudonymization

Person names are detected with a Spanish named-entity recognition model (`mrm8488/bert-spanish-cased-finetuned-ner`) and complemented with a private list of names known to the author, which is stored in `datos/raw/` so that no real name appears in this notebook. E-mail addresses, mobile phone numbers and eight-digit national ID numbers are detected with regular expressions. Every identifier is replaced by a consistent label such as `[PERSONA_3]`.

A human review of the detected entities revealed three problems, which were corrected: names split by the tokenizer (`##` fragments), titles and institutions tagged as persons (for example, *Ministro de Justicia*), and surnames that are also common words (*Carrera*, *Alegría*). For this reason, replacements are case-sensitive. Automatic pseudonymization does not replace human review, so the exported texts were also checked manually.

In [15]:
import json
import re

from transformers import pipeline

# Rebuild the original tables from the stored results, so pseudonymization always starts from raw text.
documentos = pd.DataFrame(registros)
transcripciones = pd.DataFrame(filas_asr)

# Spanish named-entity recognizer (suggested in the assignment's bonus ideas).
ner = pipeline("token-classification", model="mrm8488/bert-spanish-cased-finetuned-ner",
               aggregation_strategy="simple", device=-1)

# Real names known to the author are read from datos/raw/ (never versioned),
# so no real name is written in this notebook.
archivo_nombres = RAW / "nombres_reales.txt"
nombres_manuales = []
if archivo_nombres.exists():
    lineas_nombres = archivo_nombres.read_text(encoding="utf-8").splitlines()
    nombres_manuales = [n.strip() for n in lineas_nombres if n.strip()]


def personas_en(texto: str, tam: int = 80) -> set[str]:
    """Person names detected by the NER model, in chunks of 80 words (the model accepts 512 tokens)."""
    palabras = texto.split()
    nombres = set()
    for i in range(0, len(palabras), tam):
        for ent in ner(" ".join(palabras[i:i + tam])):
            if ent["entity_group"] == "PER" and ent["score"] > 0.8:
                nombres.add(ent["word"].strip())
    return nombres


def limpiar_nombre(nombre: str) -> str:
    """Remove sub-word markers ('##') and a leading word fragment left by the tokenizer."""
    nombre = nombre.replace("##", "").strip()
    partes = nombre.split()
    if len(partes) > 1 and partes[0][0].islower():
        partes = partes[1:]
    return " ".join(partes)


# Entities wrongly tagged as persons during the human review (titles, institutions, places, noise).
NO_SON_NOMBRES = {"Ministro de Justicia", "Fiscal de la Nación", "Derechos Humanos", "Ministro",
                  "Ramón Castilla Marquisado", "Prin", "Octa", "Arca"}

detectados = set()
for texto in list(documentos["texto_ocr"]) + list(transcripciones["texto_asr"]):
    detectados |= {limpiar_nombre(n) for n in personas_en(texto)}
detectados = {n for n in detectados
              if len(n) > 3 and re.fullmatch(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ .'-]+", n) and n not in NO_SON_NOMBRES}
detectados |= set(nombres_manuales)

# Longest names first, so full names are replaced before their parts.
orden = sorted(detectados, key=len, reverse=True)
seudonimos = {nombre: f"[PERSONA_{i}]" for i, nombre in enumerate(orden, start=1)}

# Direct identifiers that are not names.
PATRONES = {
    "[CORREO]": r"[\w.+-]+@[\w-]+\.[\w.]+",
    "[TELEFONO]": r"\b9\d{2}[\s-]?\d{3}[\s-]?\d{3}\b",
    "[DNI]": r"\b\d{8}\b",
}


def seudonimizar_texto(texto: str) -> tuple[str, int]:
    """Replace names (case-sensitive, so common words such as 'carrera' are kept) and direct identifiers."""
    total = 0
    for nombre, alias in seudonimos.items():
        texto, k = re.subn(rf"\b{re.escape(nombre)}\b", alias, texto)
        total += k
    for alias, patron in PATRONES.items():
        texto, k = re.subn(patron, alias, texto)
        total += k
    return texto, total


documentos[["texto_ocr", "reemplazos"]] = documentos["texto_ocr"].apply(
    lambda t: pd.Series(seudonimizar_texto(t)))
transcripciones[["texto_asr", "reemplazos"]] = transcripciones["texto_asr"].apply(
    lambda t: pd.Series(seudonimizar_texto(t)))

# The name-to-pseudonym table stays in datos/raw/ for the author's review and for Part 2.
(RAW / "seudonimos.json").write_text(json.dumps(seudonimos, ensure_ascii=False, indent=1), encoding="utf-8")

print(f"Distinct person names pseudonymized: {len(seudonimos)}")
print(f"Replacements in documents: {documentos['reemplazos'].sum()}")
print(f"Replacements in transcripts: {transcripciones['reemplazos'].sum()}")
print(f"Entities discarded as non-names after human review: {len(NO_SON_NOMBRES)}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: mrm8488/bert-spanish-cased-finetuned-ner
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Distinct person names pseudonymized: 45
Replacements in documents: 34
Replacements in transcripts: 4
Entities discarded as non-names after human review: 8


## 5. Cleaning

Exact duplicates are detected with a SHA-256 fingerprint of each file. Documents without any recognized text are treated as illegible, and clips in which the VAD detected no speech are discarded.

In [16]:
import hashlib


def huella(ruta: str) -> str:
    """SHA-256 fingerprint of a file, used to detect exact duplicates."""
    return hashlib.sha256(Path(ruta).read_bytes()).hexdigest()


documentos["hash"] = documentos["archivo"].map(huella)
transcripciones["hash"] = transcripciones["archivo_audio"].map(huella)

docs_duplicados = documentos.duplicated("hash")
docs_ilegibles = documentos["texto_ocr"].str.strip().eq("")
clips_duplicados = transcripciones.duplicated("hash")
clips_sin_voz = transcripciones["texto_asr"].str.strip().eq("")

print(f"Documents: {docs_duplicados.sum()} duplicates, {docs_ilegibles.sum()} illegible")
print(f"Clips: {clips_duplicados.sum()} duplicates, {clips_sin_voz.sum()} without speech")

documentos = documentos[~docs_duplicados & ~docs_ilegibles].reset_index(drop=True)
transcripciones = transcripciones[~clips_duplicados & ~clips_sin_voz].reset_index(drop=True)
print(f"Retained: {len(documentos)} documents and {len(transcripciones)} clips")

Documents: 0 duplicates, 0 illegible
Clips: 0 duplicates, 2 without speech
Retained: 25 documents and 70 clips


## 6. Exploratory data analysis

The three charts describe the composition of the corpus and the length of the extracted texts. Each title states the analytical conclusion and is computed from the data.

In [32]:
import plotly.express as px
import plotly.io as pio

# Interactive charts rendered in VS Code.
pio.renderers.default = "vscode"

# Shared visual identity: colour-blind-safe categorical palette, quiet axes, headline + subtitle.
ETIQUETAS_TIPO = {"impreso": "Printed", "foto": "Photographed", "manuscrito": "Handwritten"}
COLORES_TIPO = {"Printed": "#2a78d6", "Photographed": "#eb6834", "Handwritten": "#1baf7a"}
AZUL, TINTA, GRIS, REJILLA, FONDO = "#2a78d6", "#0b0b0b", "#52514e", "#e8e7e3", "#fcfcfb"


def estilo(fig, titulo, subtitulo):
    """Apply the common layout: bold analytical headline, grey subtitle, light grid, rounded bars."""
    fig.update_layout(
        template="simple_white",
        title=dict(text=f"<b>{titulo}</b>", x=0.01, xanchor="left", font=dict(size=19, color=TINTA),
                   subtitle=dict(text=subtitulo, font=dict(size=13, color=GRIS))),
        font=dict(family="Inter, Segoe UI, Arial", size=13, color=GRIS),
        paper_bgcolor=FONDO, plot_bgcolor=FONDO,
        margin=dict(l=70, r=30, t=110, b=70),
        bargap=0.25, barcornerradius=4,
        hoverlabel=dict(bgcolor="white", font_size=13, font_color=TINTA),
        legend=dict(orientation="h", y=-0.22, x=0.01, title_text=""),
        width=1000, height=520,
    )
    fig.update_yaxes(showgrid=True, gridcolor=REJILLA, zeroline=False, linecolor=REJILLA, ticks="")
    fig.update_xaxes(showgrid=False, linecolor="#c9c8c3", ticks="")
    return fig


documentos["n_caracteres"] = documentos["texto_ocr"].str.len()
documentos["Type"] = documentos["tipo"].map(ETIQUETAS_TIPO)
# Speaker markers are not spoken words, so they are excluded from the word count.
transcripciones["n_palabras"] = (
    transcripciones["texto_asr"].str.replace(r"\[HABLANTE_\d+\]", "", regex=True).str.split().str.len()
)

# Chart 1 — documents per type.
conteo = documentos["Type"].value_counts().reindex(COLORES_TIPO).rename_axis("Type").reset_index(name="n")
fig1 = px.bar(conteo, x="Type", y="n", color="Type", text="n", color_discrete_map=COLORES_TIPO)
fig1.update_traces(textposition="outside", showlegend=False,
                   hovertemplate="%{x}: %{y} documents<extra></extra>")
fig1.update_layout(yaxis_title="Documents", xaxis_title="", yaxis_range=[0, conteo["n"].max() * 1.25])
estilo(fig1,
       f"A balanced corpus: {conteo['n'].iloc[0]} printed, {conteo['n'].iloc[1]} photographed "
       f"and {conteo['n'].iloc[2]} handwritten documents",
       "Number of Peruvian documents per type after cleaning")
fig1.show()

# Chart 2 — minutes of audio per interview.
minutos = transcripciones.groupby("entrevista", as_index=False).agg(seg=("duracion_seg", "sum"), clips=("id", "count"))
minutos["minutos"] = minutos["seg"] / 60
minutos["Interview"] = "Interview " + minutos["entrevista"].str[-2:].str.lstrip("0")
fig2 = px.bar(minutos, x="Interview", y="minutos", text_auto=".1f", custom_data=["clips"])
fig2.update_traces(marker_color=AZUL, textposition="outside",
                   hovertemplate="%{x}<br>%{y:.1f} min in %{customdata[0]} clips<extra></extra>")
fig2.update_layout(yaxis_title="Minutes", xaxis_title="", yaxis_range=[0, minutos["minutos"].max() * 1.25])
estilo(fig2,
       f"No single voice dominates: each interview contributes "
       f"{minutos['minutos'].min():.0f}–{minutos['minutos'].max():.0f} minutes of audio",
       f"Minutes transcribed per interview · {minutos['minutos'].sum():.0f} min in "
       f"{len(transcripciones)} clips of 30 s, after removing clips without speech")
fig2.show()

# Chart 3a — length of OCR output by document type.
medianas = documentos.groupby("Type")["n_caracteres"].median()
veces = medianas["Printed"] / max(medianas["Photographed"], medianas["Handwritten"])
fig3a = px.histogram(documentos, x="n_caracteres", color="Type", nbins=20, log_x=True,
                     color_discrete_map=COLORES_TIPO, category_orders={"Type": list(COLORES_TIPO)})
fig3a.update_traces(hovertemplate="%{y} documents<extra>%{fullData.name}</extra>")
fig3a.update_layout(xaxis_title="Characters extracted by OCR (log scale)", yaxis_title="Documents")
estilo(fig3a,
       f"Printed pages yield about {veces:.0f}× more OCR text than photos or handwritten notes",
       f"Characters per document · median: printed {medianas['Printed']:,.0f}, "
       f"photographed {medianas['Photographed']:,.0f}, handwritten {medianas['Handwritten']:,.0f}")
fig3a.show()

# Chart 3b — length of ASR output per clip.
q1, mediana, q3 = transcripciones["n_palabras"].quantile([0.25, 0.5, 0.75])
fig3b = px.histogram(transcripciones, x="n_palabras", nbins=20)
fig3b.update_traces(marker_color=AZUL, hovertemplate="%{y} clips<extra></extra>")
fig3b.add_vline(x=mediana, line_dash="dot", line_color=GRIS,
                annotation_text=f"median {mediana:.0f} words", annotation_font_color=GRIS)
fig3b.update_layout(xaxis_title="Words transcribed per 30-second clip", yaxis_title="Clips")
estilo(fig3b,
       f"A conversational rhythm: half of the clips carry {q1:.0f}–{q3:.0f} words every 30 seconds",
       f"Words per clip, speaker markers excluded · n = {len(transcripciones)} clips")
fig3b.show()

## 7. Validation

The longest and shortest outputs are inspected to verify that they agree with the histograms. Because every clip lasts 30 seconds by design, clips are compared by the number of transcribed words rather than by duration.

In [21]:
# The three longest and shortest OCR outputs, to check consistency with the histogram.
cols_doc = ["id", "tipo", "n_caracteres", "confianza_media"]
display(documentos.nlargest(3, "n_caracteres")[cols_doc])
display(documentos.nsmallest(3, "n_caracteres")[cols_doc])

# All clips last 30 s by design, so clips are compared by the number of transcribed words.
cols_asr = ["id", "entrevista", "duracion_seg", "n_palabras", "n_hablantes"]
display(transcripciones.nlargest(3, "n_palabras")[cols_asr])
display(transcripciones.nsmallest(3, "n_palabras")[cols_asr])

,id,tipo,n_caracteres,confianza_media
2,doc_003,impreso,8587,0.970
4,doc_005,impreso,8154,0.955
6,doc_007,impreso,6366,0.982


,id,tipo,n_caracteres,confianza_media
8,doc_009,foto,72,0.905
11,doc_012,foto,142,0.969
16,doc_017,manuscrito,184,0.917


,id,entrevista,duracion_seg,n_palabras,n_hablantes
14,clip_015,entrevista_02,30,95,3
23,clip_024,entrevista_02,30,89,2
6,clip_007,entrevista_01,30,86,2


,id,entrevista,duracion_seg,n_palabras,n_hablantes
50,clip_053,entrevista_05,30,24,1
28,clip_029,entrevista_03,30,26,1
49,clip_052,entrevista_05,30,35,3


## 8. Export

The three CSV files contain the columns required by the assignment, followed by auxiliary columns (confidence, processing time, length and number of replacements) that are used by the dashboard. The `texto_referencia` column remains empty here and is completed in Part 2 for the evaluation sample.

In [22]:
DATOS = Path("datos")

# Required columns first; extra columns are kept for the dashboard.
cols_documentos = ["id", "archivo", "tipo", "fuente", "texto_ocr", "modelo_ocr", "texto_referencia",
                   "confianza_media", "segundos", "n_caracteres", "reemplazos"]
cols_transcripciones = ["id", "archivo_audio", "duracion_seg", "idioma", "texto_asr", "modelo_asr",
                        "n_hablantes", "texto_referencia", "entrevista", "inicio_seg", "segundos_asr",
                        "n_palabras", "reemplazos"]
documentos[cols_documentos].to_csv(DATOS / "documentos.csv", index=False, encoding="utf-8")
transcripciones[cols_transcripciones].to_csv(DATOS / "transcripciones.csv", index=False, encoding="utf-8")

# Source, licence and access instructions for every file.
URL_PERUANO = "https://busquedas.elperuano.pe/cuadernillo/NL/20261024"
filas_fuentes = []
for _, d in documentos.iterrows():
    if d["tipo"] == "impreso":
        pagina = paginas[int(d["id"].split("_")[1]) - 1]
        filas_fuentes.append({"id": d["id"], "modalidad": "documento", "fuente_url": URL_PERUANO,
                              "licencia": "Not subject to copyright (Legislative Decree 822, art. 9)",
                              "acceso": f"Download the 24/10/2026 Normas Legales issue and render page {pagina} at 200 dpi"})
    else:
        filas_fuentes.append({"id": d["id"], "modalidad": "documento", "fuente_url": "",
                              "licencia": "Author's own photograph; academic, non-commercial use",
                              "acceso": "Image not redistributed (may contain third-party content); text published after review"})
for _, c in transcripciones.iterrows():
    filas_fuentes.append({"id": c["id"], "modalidad": "audio", "fuente_url": "",
                          "licencia": "Author's own research interview, signed informed consent for academic use",
                          "acceso": (f"Confidential: cut from {c['entrevista']} at second {c['inicio_seg']}; "
                                     "only pseudonymized transcripts are published")})
fuentes = pd.DataFrame(filas_fuentes)
fuentes.to_csv(DATOS / "fuentes.csv", index=False, encoding="utf-8")

for nombre, df in [("documentos", documentos[cols_documentos]),
                   ("transcripciones", transcripciones[cols_transcripciones]),
                   ("fuentes", fuentes)]:
    print(nombre, df.shape)
display(documentos[cols_documentos].head())
display(transcripciones[cols_transcripciones].head())

documentos (25, 11)
transcripciones (70, 13)
fuentes (95, 5)


,id,archivo,tipo,fuente,texto_ocr,modelo_ocr,texto_referencia,confianza_media,segundos,n_caracteres,reemplazos
0,doc_001,datos/raw/documentos/doc_001.png,impreso,"Diario Oficial El Peruano, Normas Legales 24/1...",NORMAS LEGALES Sábado 24 de octubre de 2026 4...,PaddleOCR 3.7.0 (PP-OCRv5 mobile det + latin m...,,0.924,73.9,3076,1
1,doc_002,datos/raw/documentos/doc_002.png,impreso,"Diario Oficial El Peruano, Normas Legales 24/1...",El Peruano / Sábado 24 de octubre de 2026 NOR...,PaddleOCR 3.7.0 (PP-OCRv5 mobile det + latin m...,,0.990,119.8,4438,3
2,doc_003,datos/raw/documentos/doc_003.png,impreso,"Diario Oficial El Peruano, Normas Legales 24/1...",El Peruano / Sábado 24 de octubre de 2026 NOR...,PaddleOCR 3.7.0 (PP-OCRv5 mobile det + latin m...,,0.970,198.9,8587,8
3,doc_004,datos/raw/documentos/doc_004.png,impreso,"Diario Oficial El Peruano, Normas Legales 24/1...",El Peruano / Sábado 24 de octubre de 2026 NOR...,PaddleOCR 3.7.0 (PP-OCRv5 mobile det + latin m...,,0.976,133.5,6219,1
4,doc_005,datos/raw/documentos/doc_005.png,impreso,"Diario Oficial El Peruano, Normas Legales 24/1...",El Peruano / Sábado 24 de octubre de 2026 NOR...,PaddleOCR 3.7.0 (PP-OCRv5 mobile det + latin m...,,0.955,177.3,8154,1


,id,archivo_audio,duracion_seg,idioma,texto_asr,modelo_asr,n_hablantes,texto_referencia,entrevista,inicio_seg,segundos_asr,n_palabras,reemplazos
0,clip_001,datos/raw/audio/clips/clip_001.wav,30,es (Peruvian Spanish),[HABLANTE_1] de comunicación utiliza la biblio...,"faster-whisper small, int8, CPU, VAD on",2,,entrevista_01,60,16.9,70,3
1,clip_002,datos/raw/audio/clips/clip_002.wav,30,es (Peruvian Spanish),[HABLANTE_1] ¿Cuál es el nombre del colegio? ¿...,"faster-whisper small, int8, CPU, VAD on",3,,entrevista_01,90,18.6,40,0
2,clip_003,datos/raw/audio/clips/clip_003.wav,30,es (Peruvian Spanish),"[HABLANTE_1] bueno, para que estaba cerca de m...","faster-whisper small, int8, CPU, VAD on",1,,entrevista_01,120,21.1,81,0
3,clip_004,datos/raw/audio/clips/clip_004.wav,30,es (Peruvian Spanish),"[HABLANTE_1] hasta acá me demoraba, me sigo a ...","faster-whisper small, int8, CPU, VAD on",2,,entrevista_01,150,16.9,83,0
4,clip_005,datos/raw/audio/clips/clip_005.wav,30,es (Peruvian Spanish),[HABLANTE_1] ambiental del medio ambiental en ...,"faster-whisper small, int8, CPU, VAD on",1,,entrevista_01,180,13.5,60,0


## 9. Discard report and notes

- **Documents:** 0 duplicates and 0 illegible documents, so all 25 were retained. Before processing, two candidate photographs of English-language book pages were excluded because the corpus is limited to Peruvian documents in Spanish. They were replaced by handwritten notes.
- **Audio:** 0 duplicates. Two clips (`clip_032` and `clip_033`) contained no speech according to the VAD and were discarded, leaving **70 clips (35 minutes)**, above the required 30 minutes and 20 clips.
- **Diarization:** most clips show two speakers (interviewer and interviewee), but some report three or more. `clip_064`, for example, reports six. These counts are over-segmentations of short turns and background noise rather than additional participants, which is a known limitation of embedding clustering on 30-second segments.